# Feature Selection — отбор признаков простыми словами

Учебный ноутбук: как **убрать лишние** столбцы, оставить полезные и не словить **утечку** при отборе.

Три больших семьи методов:

| Семья | Идея | Примеры |
|-------|------|---------|
| **Filter** | смотрим на данные / y **без** (или почти без) финальной модели | variance, corr, VIF, MI, ANOVA, χ² |
| **Embedded** | отбор **внутри** обучения модели | Lasso (L1), Elastic Net, tree importance |
| **Wrapper** | крутим модель, ищем лучший **набор** | RFE, SFS |
| **+ post-hoc** | после обучения: «насколько упало качество» | **Permutation Importance** |

**Библиотеки:** `numpy`, `pandas`, `matplotlib`, `scikit-learn` (+ опционально `statsmodels` для VIF).

Запускайте **сверху вниз**.

---

## План

1. [Зачем отбирать признаки](#why)
2. [Filter: VarianceThreshold](#var)
3. [Ковариация → корреляция](#corr)
4. [Мультиколлинеарность и VIF](#vif)
5. [Mutual Information](#mi)
6. [ANOVA F-test и Chi-Square](#anova)
7. [Embedded: Lasso (L1), Ridge (L2), Elastic Net](#l1)
8. [Геометрия L1 vs L2](#geom)
9. [Tree Feature Importance](#tree)
10. [Permutation Importance](#perm)
11. [Wrapper: RFE и Sequential Feature Selection](#wrap)
12. [Практика: что используют чаще](#prac)
13. [Шпаргалка](#итог)


<a id="why"></a>
## 1. Зачем отбирать признаки?

1. Меньше шума → иногда **выше** качество.  
2. Быстрее обучение и предсказание.  
3. Проще **интерпретировать** модель.  
4. Для линейных моделей — борьба с **мультиколлинеарностью**.

**Важно:** любой отбор, который смотрит на **y** (MI, ANOVA, Lasso, RFE…), нужно делать **только на train** / **внутри Pipeline+CV**, иначе утечка.

```text
Train → отобрали признаки → теми же правилами transform Test
```


<a id="var"></a>
## 2. Filter №1 — Variance Threshold

Если признак **почти не меняется**, он мало что рассказывает модели → кандидат на удаление.

Пример: почти все нули, редкая единица → **маленькая** дисперсия.

$$
\mathrm{Var}(X) = \frac{1}{n}\sum_i (x_i - \bar x)^2
$$

(в sklearn у `VarianceThreshold` — дисперсия с делением на $n$, population-style.)

```python
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.0)  # только константы
X_new = selector.fit_transform(X)
```

| threshold | Эффект |
|-----------|--------|
| `0` | только **константные** (самый частый старт) |
| `0.01` | ещё и «почти константные» |

### Плюсы / минусы

**+** мгновенно, просто, без модели.  
**−** **не смотрит на y**. Редкий, но **критичный** признак (болезнь = 1 у 0.1% строк) может иметь малую дисперсию — **нельзя** слепо резать.

### Когда

Самый первый шаг очистки: константы / почти константы.  
Дальше — методы, которые смотрят на **полезность** или **дубли**.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

rng = np.random.default_rng(0)
X = pd.DataFrame({
    "const": 5.0,                                    # константа
    "almost": rng.choice([0, 1], size=200, p=[0.99, 0.01]),  # почти константа
    "useful": rng.normal(0, 1, size=200),
    "noise": rng.normal(0, 0.3, size=200),
})
print("Дисперсии:")
print(X.var(ddof=0).round(4))  # как у VarianceThreshold

for thr in [0.0, 0.01, 0.05]:
    sel = VarianceThreshold(threshold=thr)
    sel.fit(X)
    kept = X.columns[sel.get_support()].tolist()
    print(f"threshold={thr}: оставляем {kept}")


<a id="corr"></a>
## 3. Ковариация и корреляция

### Ковариация: «меняются ли вместе?»

$$
\mathrm{Cov}(X,Y) = \frac{1}{n}\sum_i (x_i-\bar x)(y_i-\bar y)
$$

| Знак | Смысл |
|------|--------|
| $>$ 0 | чаще растут вместе |
| $<$ 0 | один ↑, другой ↓ |
| ≈ 0 | линейной совместности почти нет |

**Минус:** зависит от **масштаба** (рубли vs тысячи рублей) → нельзя сравнивать «силу» разных пар.

### Корреляция Пирсона — нормализованная ковариация

$$
r = \frac{\mathrm{Cov}(X,Y)}{\sigma_X \sigma_Y}, \quad -1 \le r \le 1
$$

| $r$ | Смысл |
|-----|--------|
| $\approx 1$ | сильная прямая **линейная** связь |
| $\approx -1$ | сильная обратная линейная |
| $\approx 0$ | **линейной** связи нет (нелинейная $y=x^2$ может быть!) |

### Отбор по корреляции между признаками

Если $|r| > 0.9$ или $0.95$ — признаки почти дубли (рост в см и в м).  
Оставляют более понятный / менее шумный / важный для бизнеса.

**Pipeline?** Обычно на этапе **EDA**, не как шаг модели.  
Но **не** подглядывайте в test при принятии решений «что выкинуть» для финальной оценки — лучше решать на train.

### Где критично

Линейные / логистические модели — **да** (мультиколлинеарность).  
Деревья / бустинги — высокая corr **не** приговор (выберут один из пары).


In [ ]:
import matplotlib.pyplot as plt

# Ковариация зависит от масштаба, корреляция — нет
age = np.array([20., 30., 40.])
inc_k = np.array([40., 50., 60.])          # тысячи
inc_rub = inc_k * 1000                    # рубли

def cov(a, b):
    return np.mean((a - a.mean()) * (b - b.mean()))

def corr(a, b):
    return cov(a, b) / (a.std() * b.std())

print(f"Cov(age, income_тыс)  = {cov(age, inc_k):.1f}")
print(f"Cov(age, income_руб)  = {cov(age, inc_rub):.1f}  ← раздулась")
print(f"Corr(age, income_тыс) = {corr(age, inc_k):.3f}")
print(f"Corr(age, income_руб) = {corr(age, inc_rub):.3f}  ← та же")

# Нелинейность: y = x^2
x = np.linspace(-2, 2, 200)
y = x ** 2
print(f"\nCorr(x, x²) ≈ {corr(x, y):.3f}  (почти 0, хотя связь жёсткая!)")

# Матрица corr
dfc = pd.DataFrame({
    "height_cm": rng.normal(170, 10, 150),
})
dfc["height_m"] = dfc["height_cm"] / 100
dfc["noise"] = rng.normal(0, 1, 150)
dfc["y"] = 0.5 * dfc["height_cm"] + rng.normal(0, 3, 150)
print("\ncorr matrix:")
print(dfc.corr().round(2))


<a id="vif"></a>
## 4. Мультиколлинеарность и VIF

**Мультиколлинеарность** — один признак **почти объясняется** другими.

Тогда у линейной модели много «одинаково хороших» наборов коэффициентов →  
коэффициенты **прыгают**, плохо интерпретируются.

Деревья: просто берут один из похожих признаков для split — почти не страдают.

### Пара корреляции не всё видит

$A \approx B + C$ при умеренных парных corr — корреляционная матрица может **не** кричать.  
Нужен **VIF**.

### VIF (Variance Inflation Factor)

Для каждого признака $X_j$ строят регрессию **на все остальные** и смотрят $R^2_j$:

$$
\boxed{\mathrm{VIF}_j = \dfrac{1}{1 - R^2_j}}
$$

> **Исправление:** в части конспектов ошибочно пишут $\mathrm{VIF}=1-R^2$.  
> Правильно: **$1/(1-R^2)$**.  
> При $R^2=0.9$ → VIF $=10$; при $R^2=0$ → VIF $=1$.

| VIF | Интерпретация |
|-----|----------------|
| ≈ 1 | уникален |
| 1–5 | обычно ок |
| 5–10 | насторожиться |
| > 10 | сильная мультиколлинеарность (иногда порог 5) |

**Процесс:** посчитали VIF → удалили худший → **пересчитали** → пока все приемлемы.

**Средства:** `statsmodels.stats.outliers_influence.variance_inflation_factor`  
или ручной расчёт через $R^2$.

**Pipeline?** Нет, это диагностика **до** линейной модели (на train).


In [ ]:
from sklearn.linear_model import LinearRegression

def vif_scores(X_df: pd.DataFrame) -> pd.Series:
    """VIF_j = 1 / (1 - R²_j), где X_j ~ остальные."""
    vifs = {}
    cols = list(X_df.columns)
    for j, col in enumerate(cols):
        y_j = X_df[col].values
        X_rest = X_df.drop(columns=[col]).values
        if X_rest.shape[1] == 0:
            vifs[col] = 1.0
            continue
        r2 = LinearRegression().fit(X_rest, y_j).score(X_rest, y_j)
        r2 = min(r2, 0.999999)  # численная защита
        vifs[col] = 1.0 / (1.0 - r2)
    return pd.Series(vifs, name="VIF").sort_values(ascending=False)

# A ≈ B + C → высокий VIF у всех троих
n = 200
B = rng.normal(0, 1, n)
C = rng.normal(0, 1, n)
A = B + C + rng.normal(0, 0.05, n)  # почти B+C
Z = rng.normal(0, 1, n)             # независимый
X_mc = pd.DataFrame({"A": A, "B": B, "C": C, "Z": Z})
print("Парные corr (скромные у пар, но A≈B+C):")
print(X_mc.corr().round(2))
print("\nVIF:")
print(vif_scores(X_mc).round(2))
print("\nПосле удаления A:")
print(vif_scores(X_mc.drop(columns=["A"])).round(2))


<a id="mi"></a>
## 5. Mutual Information (взаимная информация)

**Вопрос:** сколько информации о **y** несёт признак $X$?  
Насколько знание $X$ уменьшает неопределённость о $y$?

$$
\mathrm{MI}(X;Y) = H(Y) - H(Y\mid X)
$$

| | Корреляция | MI |
|--|------------|-----|
| Связь | в основном **линейная** | **и нелинейная** ($y=x^2$) |
| y | нет (между признаками) / да (с y — отдельный corr) | **с y** |
| Направление | знак $r$ | только «есть связь», без знака |

```python
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression, SelectKBest

SelectKBest(score_func=mutual_info_classif, k=10).fit_transform(X, y)
```

**Pipeline + CV — да**, если отбор входит в модель (иначе подглядывание в valid).

MI **не** заменяет corr/VIF:  
- corr/VIF → «не **дубли** ли признаки друг друга?»  
- MI → «полезен ли для **y**?»


In [ ]:
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif, SelectKBest
from sklearn.datasets import make_classification, make_regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Нелинейность: MI vs corr
x = rng.uniform(-3, 3, 500)
y_nl = x ** 2 + rng.normal(0, 0.3, 500)
print(f"Corr(x, x²+noise) ≈ {np.corrcoef(x, y_nl)[0,1]:.3f}")
print(f"MI(x, x²+noise)   ≈ {mutual_info_regression(x.reshape(-1,1), y_nl, random_state=0)[0]:.3f}")

# SelectKBest + Pipeline
X_c, y_c = make_classification(n_samples=400, n_features=20, n_informative=5,
                               n_redundant=5, random_state=0)
pipe = Pipeline([
    ("sel", SelectKBest(mutual_info_classif, k=8)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
acc = cross_val_score(pipe, X_c, y_c, cv=5, scoring="accuracy").mean()
print(f"\nCV accuracy с SelectKBest(MI, k=8): {acc:.3f}")


<a id="anova"></a>
## 6. ANOVA F-test и Chi-Square

### ANOVA / `f_classif` (классификация, **числовые** X)

Отличаются ли **средние** признака **между классами**?

$$
F \approx \frac{\text{разброс между классами}}{\text{разброс внутри классов}}
$$

Большой $F$ → классы хорошо разделены по среднему признака.

```python
from sklearn.feature_selection import f_classif, SelectKBest
SelectKBest(f_classif, k=10)
```

Для **регрессии** — `f_regression` (связь признака с непрерывным y).

**Минус:** смотрит в основном на **средние** / линейный сигнал; сложную нелинейность может пропустить (MI универсальнее).

### Chi-Square `chi2` (классификация, **неотрицательные** / категории)

«Связан ли категориальный (или счётный) признак с классом?»

$$
\chi^2 = \sum \frac{(O - E)^2}{E}
$$

**Ограничение:** значения **≥ 0**. После `StandardScaler` (есть минусы) — **нельзя**.  
Сначала selection → потом scale, или не смешивать.

```python
SelectKBest(chi2, k=10)  # после One-Hot часто
```


In [ ]:
from sklearn.feature_selection import f_classif, chi2

X_bin, y_bin = make_classification(n_samples=300, n_features=6, n_informative=2,
                                   n_redundant=0, random_state=1)
# сделаем один признак «возраст-подобным» разделителем
X_bin[:, 0] = y_bin * 3 + rng.normal(0, 0.5, size=len(y_bin))

F, p = f_classif(X_bin, y_bin)
print("F-scores:", np.round(F, 2))
print("Лучший признак по ANOVA: столбец", int(np.argmax(F)))

# chi2 нужен non-negative
X_pos = np.abs(X_bin)
chi, _ = chi2(X_pos, y_bin)
print("chi2 scores:", np.round(chi, 2))


<a id="l1"></a>
## 7. Embedded: Lasso (L1), Ridge (L2), Elastic Net

### Зачем регуляризация

Большие по модулю коэффициенты → модель **дергается** от шума (переобучение).  
Штраф за «большие $w$» стабилизирует решение.

### Lasso = MSE + L1-штраф → **умеет занулять** $w$ → Feature Selection

$$
\min_w \; \mathrm{MSE} + \lambda \sum_j |w_j|
$$

В sklearn: `Lasso(alpha=...)`, где `alpha` ≈ $\lambda$.  
`alpha=0` → обычная линейная регрессия (численно лучше не ставить ровно 0).

Коэффициенты $=0$ → признак **выключен**.

**Почти всегда:** `StandardScaler` **до** Lasso (иначе штраф нечестен к масштабу).

```python
Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(alpha=0.1))])
```

### Ridge = MSE + L2-штраф → **НЕ** Feature Selection

$$
\min_w \; \mathrm{MSE} + \lambda \sum_j w_j^2
$$

Коэффициенты **сжимаются**, но почти **никогда не 0**.  
Отлично при мультиколлинеарности и переобучении, но **не** отбирает признаки.

### Elastic Net = L1 + L2

```python
ElasticNet(alpha=0.1, l1_ratio=0.7)  # l1_ratio=1 ≈ Lasso, 0 ≈ Ridge
```

Мягче Lasso на **группах** коррелированных признаков (может оставить оба, уменьшив веса).

### L1 vs L2 — грубое правило

| Хочешь | Бери |
|--------|------|
| Много мусорных признаков, нужен отбор | **L1 / Lasso** |
| Все признаки «примерно нужны», нужна стабильность | **L2 / Ridge** |
| Много признаков **и** сильные корреляции | **Elastic Net** |

### LogisticRegression

`penalty="l1", solver="saga"` (или `liblinear`) — L1-отбор для классификации.


In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# y зависит только от первых 3 признаков + шум
n, p = 150, 12
X_e = rng.normal(size=(n, p))
true_w = np.array([3.0, -2.0, 1.5] + [0.0] * (p - 3))
y_e = X_e @ true_w + rng.normal(0, 0.5, n)
names = [f"f{i}" for i in range(p)]

def show_coefs(model, title):
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    pipe.fit(X_e, y_e)
    coef = pipe.named_steps["model"].coef_
    s = pd.Series(coef, index=names)
    print(title)
    print(s.round(3).to_string())
    print(f"  ненулевых: {(np.abs(s) > 1e-6).sum()}\n")

show_coefs(Lasso(alpha=0.15, max_iter=10000), "Lasso (L1) — много нулей:")
show_coefs(Ridge(alpha=1.0), "Ridge (L2) — все ненулевые, сжаты:")
show_coefs(ElasticNet(alpha=0.15, l1_ratio=0.7, max_iter=10000), "ElasticNet:")


<a id="geom"></a>
## 8. Геометрия: почему L1 зануляет, а L2 — нет

На плоскости $(w_1, w_2)$:

1. **Без регуляризации** минимум MSE — центр эллипсов (линии одинакового MSE).  
2. **Ограничение** на размер коэффициентов:  
   - **L2:** $w_1^2 + w_2^2 \le t$ → **круг** (шар);  
   - **L1:** $|w_1| + |w_2| \le t$ → **ромб**.  
3. Ищем **наименьший** эллипс (меньшая ошибка), который **ещё касается** допустимой области.

| | L2 (круг) | L1 (ромб) |
|--|-----------|-----------|
| Касание | обычно **не** на оси | часто в **вершине** ромба |
| Вершина | — | одна из координат **= 0** |
| Эффект | оба $w$ малы, оба ≠ 0 | один $w$ часто **обнулён** |

Мы **фиксируем** размер ромба/круга (силу регуляризации $t$ / $\lambda$)  
и ищем минимально возможную ошибку при этом ограничении.

В высокой размерности у L1-«шара» много **углов** → ещё чаще разреженные решения.  
Это и есть геометрическая причина Feature Selection у Lasso.


In [ ]:
# Схема: эллипсы MSE + круг L2 + ромб L1
from matplotlib.patches import Circle, Polygon, FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

def draw_ellipses(ax):
    # концентрические «эллипсы» MSE (для наглядности — круги, сплюснутые)
    for s, c in zip([0.5, 1.0, 1.6, 2.3, 3.0], plt.cm.Blues(np.linspace(0.35, 0.9, 5))):
        ell = plt.matplotlib.patches.Ellipse((0.8, 0.9), width=2.2*s, height=1.3*s,
                                              fill=False, edgecolor=c, lw=1.5)
        ax.add_patch(ell)
    ax.plot(0.8, 0.9, "ko", label="min MSE без reg")

# L1 ромб
ax = axes[0]
draw_ellipses(ax)
t = 2.0
rombus = np.array([[t, 0], [0, t], [-t, 0], [0, -t]])
ax.add_patch(Polygon(rombus, closed=True, fill=False, edgecolor="green", lw=2.5))
ax.plot(0, t, "ro", markersize=8, label="типичное касание (w1=0)")
ax.set_title("L1: ромб |w1|+|w2|≤t — касание в вершине")
ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
ax.set_aspect("equal"); ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.legend(loc="upper right", fontsize=8); ax.set_xlabel("w1"); ax.set_ylabel("w2"); ax.grid(True, alpha=0.3)

# L2 круг
ax = axes[1]
draw_ellipses(ax)
ax.add_patch(Circle((0, 0), 2.0, fill=False, edgecolor="blue", lw=2.5))
ax.plot(1.1, 1.55, "ro", markersize=8, label="касание не на оси")
ax.set_title("L2: круг w1²+w2²≤t — оба коэффициента ≠ 0")
ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
ax.set_aspect("equal"); ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.legend(loc="upper right", fontsize=8); ax.set_xlabel("w1"); ax.set_ylabel("w2"); ax.grid(True, alpha=0.3)

plt.suptitle("Геометрия регуляризации (схема)", y=1.02)
plt.tight_layout(); plt.show()
print("L1 → углы ромба → разреженность (feature selection).")
print("L2 → гладкий круг → сжатие без нулей.")


<a id="tree"></a>
## 9. Tree Feature Importance

Дерево/лес во время обучения копит:  
насколько split по признаку **улучшал** узел (Gini/entropy ↓ или MSE ↓).

```python
model = RandomForestClassifier().fit(X_train, y_train)
pd.Series(model.feature_importances_, index=cols).sort_values(ascending=False)
```

`SelectFromModel(model, threshold="median")` — автоотбор.

### Оговорки

1. Это важность **для этой** обученной модели, не «истина мира».  
2. Непрерывные признаки с кучей порогов могут **завышаться** vs бинарные.  
3. Коррелированные признаки **делят** importance.  
4. **≠** «если убрать, метрика упадёт на 40%» → для этого **Permutation Importance**.

Масштаб не нужен. Считается из **train-процесса**.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

X_t, y_t = make_classification(n_samples=400, n_features=10, n_informative=4,
                               n_redundant=3, random_state=2)
cols = [f"x{i}" for i in range(X_t.shape[1])]
Xf = pd.DataFrame(X_t, columns=cols)

rf = RandomForestClassifier(n_estimators=200, random_state=0)
rf.fit(Xf, y_t)
imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
print("Tree feature_importances_:")
print(imp.round(3).to_string())

sel = SelectFromModel(rf, threshold="median", prefit=True)
print("\nОставлено столбцов:", sel.transform(Xf).shape[1])


<a id="perm"></a>
## 10. Permutation Importance

**После** обучения модели:

1. Перемешать **один** столбец (разрушить связь с y, сохранив распределение).  
2. Пересчитать метрику.  
3. Насколько **упало** качество → важность.

```python
from sklearn.inspection import permutation_importance
r = permutation_importance(model, X_val, y_val, n_repeats=10, random_state=0)
r.importances_mean
```

**Считать на validation/test**, не на train (иначе оптимизм).

### Плюсы / минусы

**+** любая модель; интерпретация «падение метрики».  
**−** дороже; при **двух копиях** одного сигнала каждый по отдельности может казаться «не важным» (модель переключается на близнеца).

В CV: в каждом фолде fit на train_fold → permutation на **val_fold** → усреднить importances.


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(Xf, y_t, test_size=0.3, random_state=0)
rf2 = RandomForestClassifier(n_estimators=150, random_state=1).fit(Xtr, ytr)
r = permutation_importance(rf2, Xte, yte, n_repeats=15, random_state=0, scoring="accuracy")
perm = pd.Series(r.importances_mean, index=cols).sort_values(ascending=False)
print("Permutation importance (на test):")
print(perm.round(4).to_string())
print("\nСравните порядок с tree importance — может отличаться.")


<a id="wrap"></a>
## 11. Wrapper: RFE и Sequential Feature Selection

### RFE — Recursive Feature Elimination

```text
все признаки → fit → выкинуть худший (по coef_/importance_)
→ fit → выкинуть → … → пока не останется k
```

```python
from sklearn.feature_selection import RFE
RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=5)
```

`support_`, `ranking_` (1 = вошёл в финал).

**Минус:** много переобучений модели (медленно на 100+ признаках + CV).

### SequentialFeatureSelector (SFS)

- **forward:** с пустого набора жадно **добавляем** лучший;  
- **backward:** с полного **удаляем** худший.

В отличие от RFE, SFS ориентируется на **CV-метрику** комбинаций, а не только на coef_/importance.

```python
from sklearn.feature_selection import SequentialFeatureSelector
SequentialFeatureSelector(model, n_features_to_select=5, direction="forward", cv=5)
```

**Ещё медленнее**, чем RFE. В проде реже, чем Lasso / tree / permutation.


In [ ]:
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression

Xc, yc = make_classification(n_samples=250, n_features=12, n_informative=5, random_state=3)
feat = [f"f{i}" for i in range(Xc.shape[1])]

# RFE
rfe = RFE(LogisticRegression(max_iter=2000), n_features_to_select=5)
rfe.fit(Xc, yc)
print("RFE selected:", [f for f, s in zip(feat, rfe.support_) if s])
print("RFE ranking:", dict(zip(feat, rfe.ranking_)))

# SFS forward (cv=3 для скорости демо)
sfs = SequentialFeatureSelector(
    LogisticRegression(max_iter=2000),
    n_features_to_select=5,
    direction="forward",
    cv=3,
    n_jobs=-1,
)
sfs.fit(Xc, yc)
print("SFS selected:", [f for f, s in zip(feat, sfs.get_support()) if s])


<a id="prac"></a>
## 12. Что чаще используют на практике

Грубый рейтинг «в жизни» (зависит от стека):

1. **L1 / Lasso** (и L1-logreg)  
2. **Tree feature_importances_**  
3. **Permutation Importance**  
4. **RFE**  
5. **SFS** (редко, дорого)  

**Filter-старт почти всегда:**

```text
VarianceThreshold(0)
    → EDA: corr heatmap
    → (линейные модели) VIF
    → MI / ANOVA / chi2 + SelectKBest
    → модель: Lasso или RF + importance / permutation
```

### Честный отбор в CV

```python
Pipeline([
    ("imputer", ...),
    ("selector", SelectKBest(mutual_info_classif, k=20)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
cross_val_score(pipeline, X, y, cv=5)
```


<a id="итог"></a>
## 13. Шпаргалка

| Метод | Смотрит y? | Зануляет? | Главный риск |
|-------|------------|-----------|--------------|
| VarianceThreshold | нет | «константы» | выкинуть редкий, но важный |
| Correlation | между X | удаляем вручную | только линейное |
| VIF | нет (X~X) | вручную | формула $1/(1-R^2)$ |
| MI / ANOVA / χ² | **да** | SelectKBest | leakage, если на всём X |
| Lasso L1 | да (в loss) | **да** | нужен scaling |
| Ridge L2 | да | **нет** | не FS |
| Elastic Net | да | частично | 2 гиперпараметра |
| Tree importance | да (train) | порог | смещения |
| Permutation | да (после fit) | нет | коррелированные близнецы |
| RFE / SFS | да | да | скорость |

### Главные мысли

1. Filter чистит и диагностирует; Embedded/Wrapper **под модель**.  
2. **VIF = 1/(1−R²)**, не $1-R^2$.  
3. Corr ≈ 0 ≠ «связи нет» (бывает нелинейная).  
4. L1 отбирает из‑за **геометрии ромба**; L2 — круг, без нулей.  
5. Отбор с y — только train / Pipeline+CV.


### Мини-практика

1. Найдите константный столбец через `VarianceThreshold(0)`.  
2. Покажите, что corr($x$, $x^2$)≈0, а MI — нет.  
3. Соберите мультиколлинеарность $A\approx B+C$ и посчитайте VIF.  
4. `Lasso` vs `Ridge` на данных с 3 полезными и 10 нулевыми признаками.  
5. Сравните tree importance и permutation importance.


In [ ]:
# ===== Краткая сводка на синтетике =====
print("VIF check A≈B+C: already above")
print("Lasso zeros vs Ridge: already above")
print("OK: notebook ready. Главное — не путать VIF=1/(1-R2) и не отбирать признаки на test.")


## Что делать дальше

1. Свяжите с **Feature Engineering**: сначала создаём признаки, потом отбираем.  
2. Свяжите с **валидацией**: selector внутри Pipeline.  
3. Для интерпретации коэффициентов линейной модели — corr + VIF + возможно Lasso.

### Главная мысль

> Отбор признаков — это не один алгоритм, а **конвейер здравого смысла**:  
> убрать мусор → убрать дубли → оценить пользу для y → (для линейных) стабилизировать коэффициенты.

Удачи!
